In [16]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


Importing the SparkSession Class and the function that lets us use the column function.
- `SparkSession`: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.html
- `functions`: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html

In [ ]:
AWS_ACCESS_KEY_ID = "xxxxxxxxxxxx"
AWS_SECRET_ACCESS_KEY = "xxxxxxxxxxx"
# AWS_REGION = "eu-central-1"

INPUT_PATH = "s3a://cde-spark-demo/demo/orders/"
OUTPUT_PATH = "s3a://cde-spark-demo/demo/revenue_by_region/"


Considering we need to interract with `AWS`, passing in the credentials and setting the `source` and `destination` path in `s3`.

In [27]:
spark = (
    SparkSession.builder.appName("RevenueByRegion")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    # .config("spark.hadoop.fs.s3a.endpoint.region", AWS_REGION)
    .getOrCreate()
)

This Cell basically create the Spark Application called `RevenueByRegion`
- `SparkSession.builder.appName`: This is where we input the name of your spark application https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.builder.appName.html
- `config`: Used to pass in the AWS credentials, on cloud platforms, this won't be needed as one will likely use Instance Profile.
- `getOrCreate()`: This basically saying check if we have an existing application called `RevenueByRegion`, otherwise create it.

In [28]:
orders = spark.read.parquet(INPUT_PATH)

paid_orders = orders.filter(F.col("status") == "PAID")

paid_orders.show()

+--------+------+------+------+
|order_id|region|amount|status|
+--------+------+------+------+
|     101|    EU|   200|  PAID|
|     103|    EU|   300|  PAID|
|     104|    US|   150|  PAID|
|     105|    EU|   400|  PAID|
+--------+------+------+------+



- Reading the data in s3
- Filter only the PAID status
- Show the Top 5

In [29]:
revenue_by_region = (
    paid_orders.groupBy("region")
    .agg(F.sum("amount").alias("total_revenue"))
    .orderBy("region")
)

revenue_by_region.show()

[Stage 2:>                                                          (0 + 1) / 1]

+------+-------------+
|region|total_revenue|
+------+-------------+
|    EU|          900|
|    US|          150|
+------+-------------+



- Group by region
- Doing an aggregation of the amount per region
- Using alias function to rename the column.
- Order by the region in Ascending order

In [30]:
revenue_by_region.coalesce(1).write.mode("overwrite").parquet(OUTPUT_PATH)

Write the aggregated DataFrame to s3.
- coalesce(1): This ensure the data has one partition, which generate one file in s3.
- Mode: This takes in Overwrite to ensure the file is overwritten always.
- Parquet: The file format to write in s3.

In [31]:
spark.stop()

Stopping the spark application!!!!